<a href="https://colab.research.google.com/github/BrooksNguyen/intern-vnpt-ai/blob/main/VNPT_Internship_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dự án Thực tập VNPT AI - Data Analyst
**Mục tiêu:** Tối ưu hoá dữ liệu hệ thống Chat (ETL, Time-Bucketing, NLP)

Notebook này mô phỏng toàn bộ chu trình xử lý dữ liệu từ Phase 0 đến Phase 4 (Tuần 8) trực tiếp trên Google Colab.
Thay vì dùng Docker như máy cục bộ, toàn bộ hệ thống Database & Big Data Engine (Cassandra + PySpark) sẽ được dựng thẳng trong môi trường Colab.

## 1. Cài đặt Môi trường (Cassandra & Spark)

In [ ]:
!apt-get update -qq
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget https://archive.apache.org/dist/cassandra/4.1.7/apache-cassandra-4.1.7-bin.tar.gz
!tar -xzf apache-cassandra-4.1.7-bin.tar.gz
!pip install -q pyspark==3.5.1 underthesea cassandra-driver matplotlib pandas

import os
import time
import subprocess

# Set JVM limits to prevent Colab from crashing
os.environ["MAX_HEAP_SIZE"] = "512M"
os.environ["HEAP_NEWSIZE"] = "100M"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

print("Đang khởi động Cassandra Database (khoảng 30 giây)...")
subprocess.Popen(["./apache-cassandra-4.1.7/bin/cassandra", "-R"])
time.sleep(30)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
--2026-08-21 09:33:38--  https://archive.apache.org/dist/cassandra/4.1.7/apache-cassandra-4.1.7-bin.tar.gz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 50402260 (48M) [application/x-gzip]
Saving to: ‘apache-cassandra-4.1.7-bin.tar.gz’

apache-cassandra-4. 100%[===================>]  48.07M   108KB/s    in 5m 8s   

2026-08-21 09:38:46 (160 KB/s) - ‘apache-cassandra-4.1.7-bin.tar.gz’ saved [50402260/50402260]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 17.3 MB/s eta 0:00:00
   ━━━━━

## 2. Sinh dữ liệu giả lập (Phase 0) & Khởi tạo Schema (Phase 2)
Dùng chính Cassandra làm Database đích (thay cho ScyllaDB) để demo tính năng Time-bucketing.

In [ ]:
from cassandra.cluster import Cluster
from cassandra.policies import DCAwareRoundRobinPolicy
from cassandra.concurrent import execute_concurrent_with_args
import random
import time
from datetime import datetime, timedelta

print("Đợi Cassandra khởi động hoàn toàn và nhận kết nối...")
for i in range(15):
    try:
        cluster = Cluster(['127.0.0.1'], port=9042, load_balancing_policy=DCAwareRoundRobinPolicy(local_dc='datacenter1'))
        session = cluster.connect()
        print("Kết nối Cassandra thành công!")
        break
    except Exception as e:
        print(f"Thử lại lần {i+1}/15 sau 5 giây...")
        time.sleep(5)
else:
    raise Exception("Không thể kết nối tới Cassandra. Vui lòng chạy lại Cell 1 hoặc restart runtime.")

# Schema Nguồn (Gặp Hot Partition)
session.execute("CREATE KEYSPACE IF NOT EXISTS chat_system WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1};")
session.set_keyspace("chat_system")
session.execute("""
    CREATE TABLE IF NOT EXISTS chat_table (
        room_id text, message_id timeuuid, user_id text, content text,
        msg_type text, device text, is_edited boolean, timestamp timestamp,
        PRIMARY KEY (room_id, message_id)
    ) WITH CLUSTERING ORDER BY (message_id DESC);
""")

# Schema Đích (Time-bucketing giải quyết Hot Partition)
session.execute("CREATE KEYSPACE IF NOT EXISTS chat_system_target WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1};")
session.set_keyspace("chat_system_target")
session.execute("""
    CREATE TABLE IF NOT EXISTS chat_table_bucketed (
        room_id text, bucket_id text, message_id timeuuid, user_id text,
        content text, msg_type text, device text, is_edited boolean, timestamp timestamp,
        PRIMARY KEY ((room_id, bucket_id), message_id)
    ) WITH CLUSTERING ORDER BY (message_id DESC);
""")

print("Tạo Schema thành công!")

print("Đang sinh mock data...")
session.set_keyspace("chat_system")
sample_msgs = ["Dạ cảm ơn", "Shop còn hàng k?", "Mạng lag quá", "Alo 123"]
devices = ["ios", "android", "web"]
insert_query = session.prepare("INSERT INTO chat_table (room_id, message_id, user_id, content, msg_type, device, is_edited, timestamp) VALUES (?, now(), ?, ?, ?, ?, ?, ?)")

params = []
for _ in range(3000): # Sinh 3000 tin nhắn để demo nhanh
    params.append(("room_999", f"user_{random.randint(1,100)}", random.choice(sample_msgs), "text", random.choice(devices), False, datetime.now() - timedelta(days=random.randint(0,60))))

execute_concurrent_with_args(session, insert_query, params, concurrency=100)
print("Sinh mock data hoàn tất!")

/tmp/ipykernel_1569/2495720580.py:11: DeprecationWarning: Legacy execution parameters will be removed in 4.0. Consider using execution profiles.
  cluster = Cluster(['127.0.0.1'], port=9042, load_balancing_policy=DCAwareRoundRobinPolicy(local_dc='datacenter1'))
Traceback (most recent call last):
  File "cassandra/cluster.py", line 3634, in cassandra.cluster.ControlConnection._reconnect_internal
  File "cassandra/cluster.py", line 3656, in cassandra.cluster.ControlConnection._try_connect
  File "cassandra/cluster.py", line 1635, in cassandra.cluster.Cluster.connection_factory
  File "cassandra/connection.py", line 858, in cassandra.connection.Connection.factory
  File "/usr/local/lib/python3.12/dist-packages/cassandra/io/libevreactor.py", line 269, in __init__
    self._connect_socket()
  File "cassandra/connection.py", line 963, in cassandra.connection.Connection._connect_socket
ConnectionRefusedError: [Errno 111] Tried connecting to [('127.0.0.1', 9042)]. Last error: Connection refuse

Đợi Cassandra khởi động hoàn toàn và nhận kết nối...
Thử lại lần 1/15 sau 5 giây...


Traceback (most recent call last):
  File "cassandra/cluster.py", line 3634, in cassandra.cluster.ControlConnection._reconnect_internal
  File "cassandra/cluster.py", line 3656, in cassandra.cluster.ControlConnection._try_connect
  File "cassandra/cluster.py", line 1635, in cassandra.cluster.Cluster.connection_factory
  File "cassandra/connection.py", line 858, in cassandra.connection.Connection.factory
  File "/usr/local/lib/python3.12/dist-packages/cassandra/io/libevreactor.py", line 269, in __init__
    self._connect_socket()
  File "cassandra/connection.py", line 963, in cassandra.connection.Connection._connect_socket
ConnectionRefusedError: [Errno 111] Tried connecting to [('127.0.0.1', 9042)]. Last error: Connection refused
ERROR:cassandra.cluster:Control connection failed to connect, shutting down Cluster:
Traceback (most recent call last):
  File "cassandra/cluster.py", line 1705, in cassandra.cluster.Cluster.connect
  File "cassandra/cluster.py", line 3600, in cassandra.cluste

Thử lại lần 2/15 sau 5 giây...


Traceback (most recent call last):
  File "cassandra/cluster.py", line 3634, in cassandra.cluster.ControlConnection._reconnect_internal
  File "cassandra/cluster.py", line 3656, in cassandra.cluster.ControlConnection._try_connect
  File "cassandra/cluster.py", line 1635, in cassandra.cluster.Cluster.connection_factory
  File "cassandra/connection.py", line 858, in cassandra.connection.Connection.factory
  File "/usr/local/lib/python3.12/dist-packages/cassandra/io/libevreactor.py", line 269, in __init__
    self._connect_socket()
  File "cassandra/connection.py", line 963, in cassandra.connection.Connection._connect_socket
ConnectionRefusedError: [Errno 111] Tried connecting to [('127.0.0.1', 9042)]. Last error: Connection refused
ERROR:cassandra.cluster:Control connection failed to connect, shutting down Cluster:
Traceback (most recent call last):
  File "cassandra/cluster.py", line 1705, in cassandra.cluster.Cluster.connect
  File "cassandra/cluster.py", line 3600, in cassandra.cluste

...

Thử lại lần 4/15 sau 5 giây...


Traceback (most recent call last):
  File "cassandra/cluster.py", line 3634, in cassandra.cluster.ControlConnection._reconnect_internal
  File "cassandra/cluster.py", line 3656, in cassandra.cluster.ControlConnection._try_connect
  File "cassandra/cluster.py", line 1635, in cassandra.cluster.Cluster.connection_factory
  File "cassandra/connection.py", line 858, in cassandra.connection.Connection.factory
  File "/usr/local/lib/python3.12/dist-packages/cassandra/io/libevreactor.py", line 269, in __init__
    self._connect_socket()
  File "cassandra/connection.py", line 963, in cassandra.connection.Connection._connect_socket
ConnectionRefusedError: [Errno 111] Tried connecting to [('127.0.0.1', 9042)]. Last error: Connection refused
ERROR:cassandra.cluster:Control connection failed to connect, shutting down Cluster:
Traceback (most recent call last):
  File "cassandra/cluster.py", line 1705, in cassandra.cluster.Cluster.connect
  File "cassandra/cluster.py", line 3600, in cassandra.cluste

Thử lại lần 5/15 sau 5 giây...


Kết nối Cassandra thành công!


Tạo Schema thành công!
Đang sinh mock data...
Sinh mock data hoàn tất!


## 3. PySpark ETL Migration (Phase 3)

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.datastax.spark:spark-cassandra-connector_2.12:3.5.0 pyspark-shell'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, date_format

spark = SparkSession.builder \
    .appName("ETL_Colab") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.cassandra.connection.host", "127.0.0.1") \
    .config("spark.cassandra.output.batch.size.bytes", "65536") \
    .config("spark.cassandra.output.concurrent.writes", "10") \
    .config("spark.driver.memory", "1g") \
    .config("spark.executor.memory", "1g") \
    .getOrCreate()

print("Đọc DataFrame từ Cassandra Nguồn...")
df_source = spark.read.format("org.apache.spark.sql.cassandra").options(table="chat_table", keyspace="chat_system").load()

# Transform: Thêm bucket_id yyyy-MM
df_transformed = df_source.withColumn("bucket_id", date_format(col("timestamp"), "yyyy-MM"))
df_transformed.show(5)

print("Thực thi luồng Ghi sang CSDL Đích...")
df_transformed.write.format("org.apache.spark.sql.cassandra") \
    .options(table="chat_table_bucketed", keyspace="chat_system_target") \
    .mode("append").save()
print("ETL Hoàn tất 100%!")

Đọc DataFrame từ Cassandra Nguồn...
+--------+--------------------+----------------+-------+---------+--------+--------------------+-------+---------+
| room_id|          message_id|         content| device|is_edited|msg_type|           timestamp|user_id|bucket_id|
+--------+--------------------+----------------+-------+---------+--------+--------------------+-------+---------+
|room_999|6cb4c828-9d44-11f...|       Dạ cảm ơn|    web|    false|    text|2026-07-28 09:41:...|user_77|  2026-07|
|room_999|6cb4c81e-9d44-11f...|       Dạ cảm ơn|    ios|    false|    text|2026-08-01 09:41:...|user_54|  2026-08|
|room_999|6cb4c814-9d44-11f...|Shop còn hàng k?|android|    false|    text|2026-07-05 09:41:...|user_91|  2026-07|
|room_999|6cb4c80a-9d44-11f...|       Dạ cảm ơn|android|    false|    text|2026-08-17 09:41:...|user_65|  2026-08|
|room_999|6cb4c800-9d44-11f...|Shop còn hàng k?|android|    false|    text|2026-07-16 09:41:...|user_71|  2026-07|
+--------+--------------------+-------------

## 4. NLP Text Preprocessing (Phase 4)

In [ ]:
import re
from underthesea import word_tokenize

EMOTICONS_PATTERN = re.compile(r'(=[)(]+|:\)|:\(|<3|\?{2,}|!{2,})')
URL_PATTERN = re.compile(r'http[s]?://\S+|www\.\S+')
PUNCT_PATTERN = re.compile(r'[^\w\s]', flags=re.UNICODE)
SPACES_PATTERN = re.compile(r'\s+')

def clean_text(text):
    if not text: return ""
    text = text.lower()
    text = URL_PATTERN.sub('', text)

    emoticons_found = EMOTICONS_PATTERN.findall(text)
    for i, emo in enumerate(emoticons_found):
        text = text.replace(emo, f" EMO_{i} ", 1)

    text = PUNCT_PATTERN.sub('', text)
    text = SPACES_PATTERN.sub(' ', text).strip()

    tokens = word_tokenize(text, format="list")

    # Lọc stopwords đơn giản
    stopwords = {"thì", "là", "mà", "bị", "được", "quá", "cho", "hỏi"}
    cleaned_tokens = [t for t in tokens if t.startswith("EMO_") or (t not in stopwords and len(t) > 1)]

    final_text = " ".join(cleaned_tokens)
    for i, emo in enumerate(emoticons_found):
        final_text = final_text.replace(f"EMO_{i}", emo)

    return final_text

test_msgs = [
    "Mạng VNPT dạo này lag quá... https://vnpt.com.vn",
    "Cho e hỏi chi phí lắp wifi bao nhiêu ạ??? =)))"
]
for msg in test_msgs:
    print("RAW:", msg)
    print("CLN:", clean_text(msg), "\n")

RAW: Mạng VNPT dạo này lag quá... https://vnpt.com.vn
CLN: mạng vnpt dạo này lag 

RAW: Cho e hỏi chi phí lắp wifi bao nhiêu ạ??? =)))
CLN: e hỏi chi phí lắp wifi bao nhiêu ??? =))) 

